# Cover Studio · Windows 10/11 本地版（RTX 3060 6GB / CUDA 12.6 一键部署版）

将原始 Colab 笔记本重构为 Windows 本地一键启动方案。无需 Google Drive、Colab 代理或 Linux 环境。

- **推荐方式**：直接双击同目录下的 `01_install.bat`（一键安装）、`02_start.bat`（一键启动）、`03_toolbox.bat`（工具箱与性能调节）。
- **Notebook 方式**：在 Python 3.12 内核中按顺序运行下方单元格即可完成安装、自检、模型预下载与服务启动。


In [ ]:
# ① 检查环境：需要 Windows + Python 3.12 内核
import sys, os, subprocess
from pathlib import Path
BASE = Path.cwd()
if not (BASE / "windows_cover_studio.py").exists():
    raise FileNotFoundError("请在解压后的 CoverStudio_Win10_RTX3060_6GB 文件夹中打开此 Notebook")
print("Python:", sys.version.split()[0])
print("工作目录:", BASE)
subprocess.run([sys.executable, str(BASE / "windows_cover_studio.py"), "check"], check=True)

In [ ]:
# ② 第一次运行：自动下载安装依赖（包含 CUDA 12.9 驱动可用的 torch 2.13.0 + cu126）
# 可以不执行本格，改为双击 01_install.bat。
import subprocess, sys
subprocess.run([sys.executable, str(BASE / "windows_cover_studio.py"), "install"], check=True)

### 启动本地服务器

- `TRANSCRIBE_ON_CPU = False`：音频转录 + 翻弹生成均使用 GPU（默认，支持通过 `03_toolbox.bat` 调节 batch=1~4）。
- `TRANSCRIBE_ON_CPU = True`：音频转录在 CPU 上运行，翻弹生成仍用 GPU（适合 6GB 显存遇到转录 OOM 时使用）。

启动成功后在浏览器访问：**http://127.0.0.1:8100**。


In [ ]:
# ③ 启动服务；另开浏览器访问 http://127.0.0.1:8100
import subprocess, sys, socket, time, os
TRANSCRIBE_ON_CPU = False
if "SERVER" in globals() and SERVER is not None and SERVER.poll() is None:
    print("服务器已由本 Notebook 启动，PID:", SERVER.pid)
else:
    mode = "transcribe-cpu" if TRANSCRIBE_ON_CPU else "gpu"
    LOGFILE = BASE / "logs" / "notebook_launcher.log"
    LOGFILE.parent.mkdir(exist_ok=True)
    SERVER_LOG = LOGFILE.open("a", encoding="utf-8")
    SERVER = subprocess.Popen(
        [sys.executable, str(BASE / "windows_cover_studio.py"), "start", "--mode", mode],
        cwd=BASE, stdout=SERVER_LOG, stderr=subprocess.STDOUT,
        creationflags=getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0),
    )
    print("启动脚本 PID:", SERVER.pid, "日志:", LOGFILE)
print("本地地址: http://127.0.0.1:8100")
for _ in range(30):
    if SERVER.poll() is not None:
        print("启动进程提前退出。请检查 logs/notebook_launcher.log")
        break
    try:
        with socket.create_connection(("127.0.0.1", 8100), timeout=1):
            print("✓ 端口已就绪，请打开上述网址")
            break
    except OSError:
        time.sleep(1)
else:
    print("仍未监听端口；先查看日志，首次下载 UI 可能需要更久")

In [ ]:
# ④ 查看日志（适合排错）
from pathlib import Path
for name in ["notebook_launcher.log", "cover_studio.log"]:
    path = BASE / "logs" / name
    if path.exists():
        print("\n=====", path, "=====")
        print("\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-50:]))

In [ ]:
# ⑤ 停止本 Notebook 启动的服务
# 注意：Windows 上需要同时关闭启动脚本及其子进程，否则 8100 端口可能残留。
import subprocess
if "SERVER" in globals() and SERVER is not None and SERVER.poll() is None:
    subprocess.run(["taskkill", "/PID", str(SERVER.pid), "/T", "/F"], check=False)
    SERVER.wait(timeout=10)
    print("已停止服务")
else:
    print("没有正在运行的 Notebook 启动进程")
if "SERVER_LOG" in globals() and not SERVER_LOG.closed:
    SERVER_LOG.close()